# Binary grain vs background (SwinV2 + UPerNet, 5-fold CV)

This notebook is separate from `colab_run_pipeline_221.ipynb`. It trains **binary** segmentation:

- **Class 0 (non-grain / background):** original multiclass label `0`.
- **Class 1 (grain):** any original label in `1`–`15` (bivalves, micrite, cement, …, brachiopod).
- **Ignore:** original value `255` stays ignored in the loss.

Multiclass masks are converted **on the fly** in the dataset (no duplicate mask files written).

Training uses the same **SwinV2-Tiny + UPerNet** setup as the multiclass pipeline (`num_labels=2`), **5-fold** cross-validation over image/mask pairs, and logs **pixel accuracy**, **mean IoU**, and **per-class IoU** each validation epoch. Summary metrics are saved to `cv_summary.json` under `--output_dir`.

In [ ]:
# Install dependencies (Colab)
!pip -q install --upgrade transformers tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

# Repo on Drive (update if your clone lives elsewhere)
REPO_ROOT = Path('/content/drive/My Drive/Payne_lab_swin_transformer')

IMG_DIR = '/content/drive/My Drive/Petrographic images_ML work/labelled images_PS/labelled images_PS/my_dataset/img'
MASK_DIR = '/content/drive/My Drive/Petrographic images_ML work/labelled images_PS/labelled images_PS/my_dataset/masks_machine'

# Outputs for binary CV runs (separate from multiclass OUT_ROOT in the other notebook)
OUT_BINARY = '/content/drive/My Drive/Petrographic images_ML work/model_outputs_binary_cv'

print('Repo exists:', REPO_ROOT.exists())
print('IMG dir exists:', Path(IMG_DIR).exists())
print('MASK dir exists:', Path(MASK_DIR).exists())

## Smoke test

Builds dataloaders and checks paired samples (no training).

In [ ]:
import os
from pathlib import Path

os.chdir(REPO_ROOT)

cmd = (
    "python code/model_training_pipeline/swin_binary_segmentation_221.py "
    f'--img_dir "{IMG_DIR}" '
    f'--mask_dir "{MASK_DIR}" '
    f'--output_dir "{OUT_BINARY}/smoke" '
    "--no_train"
)
print(cmd)
os.system(cmd)

## Full 5-fold cross-validation

Runs **five** trainings (one per fold). Each fold writes `fold_k/best_upernet_swinv2_binary.pth`, `fold_k/val_metrics.csv`, and a combined `cv_summary.json` under `OUT_BINARY`.

Optional: after SSL pretraining from the other notebook, set `SSL_BACKBONE` to your `ssl_swinv2_best.pth` path and add `--backbone_checkpoint "$SSL_BACKBONE"` to the command below.

In [ ]:
import os
from pathlib import Path

os.chdir(REPO_ROOT)

# Optional: initialize backbone from SSL checkpoint (same format as multiclass finetune)
SSL_BACKBONE = "/content/drive/My Drive/Petrographic images_ML work/model_outputs_221/ssl_full/ssl_swinv2_best.pth"
ssl_arg = f'--backbone_checkpoint "{SSL_BACKBONE}"' if Path(SSL_BACKBONE).exists() else ""
print("SSL backbone:", "ON" if ssl_arg else "OFF (ImageNet-only backbone)")

cmd = (
    "python code/model_training_pipeline/swin_binary_segmentation_221.py "
    f'--img_dir "{IMG_DIR}" '
    f'--mask_dir "{MASK_DIR}" '
    f'--output_dir "{OUT_BINARY}/cv5" '
    "--n_folds 5 "
    "--epochs 20 "
    "--batch_size 2 "
    "--crop 512 "
    "--num_workers 2 "
    f"{ssl_arg}"
)
print(cmd)
os.system(cmd)